# DP-750 · Module 12 · Development lifecycle demos
**Git folders → branches and pull requests → tests → bundle files → deploy with the CLI**

| # | Demo | Runs in | You'll see |
|---|---|---|---|
| 1 | Git folders | Workspace + your Git provider | The project files show up as changes. First **Commit & Push** |
| 2 | Branches, pull requests, conflicts | Workspace + your Git provider | A diff, a merged pull request, a conflict solved by keeping both changes |
| 3 | Testing pyramid | This notebook | 3 unit tests pass, one deliberate failure, the integration test passes, UAT accepted |
| 4 | Bundle configuration | This notebook | `databricks.yml` and the job YAML, line by line |
| 5 | Deploy with the CLI | Your laptop | validate → plan → deploy → summary. `[dev you] sales_job` with a paused schedule |

- **Standalone:** the setup cells create a schema with one sample table and a small project folder `sales_demo/` next to this notebook (code, tests, bundle YAML).
- **External services:** demos 1–2 need GitHub or Azure DevOps with an empty repository. Demo 5 needs the Databricks CLI on your laptop. Demos 3–4 need neither.
- **Compute:** Serverless. **Run All** works top to bottom. The external steps are clicks and commands outside the notebook.
- **Doing demos 1–2?** Put this notebook in a Git folder *before* you run the setup (demo 1, step 3).

## Setup - XYZ
Installs pytest (this restarts Python), then creates the schema, the sample table and the project files.
Detached after a break? Run the three setup cells again. Existing project files are kept, so your Git demo edits survive.

In [0]:
%pip install -q pytest

In [0]:
import json, os, re, sys

import pytest
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0], "Catalog")
dbutils.widgets.text("schema", "m12_demo", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
if CATALOG in ("hive_metastore", "spark_catalog"):
    raise ValueError("Pick a Unity Catalog catalog in the 'catalog' widget.")

w = WorkspaceClient()
HOST = w.config.host
USER = w.current_user.me().user_name
PROJECT = os.path.join(os.getcwd(), "sales_demo")   # next to this notebook
OVERWRITE = False                                    # True rewrites every project file (undoes Git demo edits)

# The test schema: one sample table, 5 sales, 2 of them cancelled
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
TEST_TABLE = f"`{CATALOG}`.`{SCHEMA}`.sales_sample"
spark.createDataFrame(
    [(1, 100.0, "PAID"), (2, 250.0, "paid"), (3, 80.0, " Cancelled "), (4, 40.0, "PAID "), (5, 20.0, "CANCELLED")],
    "sale_id INT, net DOUBLE, status STRING",
).write.mode("overwrite").saveAsTable(TEST_TABLE)
os.environ["TEST_TABLE"] = TEST_TABLE   # the integration test reads this
sys.path.insert(0, PROJECT)             # lets this notebook import src.sales too
sys.dont_write_bytecode = True          # no .pyc files in the workspace


def write(path, text):
    """Writes one project file, unless it exists already (from Git or an earlier run)."""
    full = os.path.join(PROJECT, path)
    if not OVERWRITE and (os.path.exists(full) or os.path.exists(os.path.splitext(full)[0])):
        return "kept"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(text.lstrip("\n"))
    return "written"


def show(path):
    """Prints a project file with line numbers."""
    print(f"── {path} ──")
    with open(os.path.join(PROJECT, path)) as f:
        for number, line in enumerate(f, 1):
            print(f"{number:3}  {line}", end="")
    print()


def show_tree():
    """Prints the project folder like the tree command."""
    for root, dirs, files in os.walk(PROJECT):
        dirs[:] = sorted(d for d in dirs if d not in ("__pycache__", ".pytest_cache", ".databricks"))
        depth = root[len(PROJECT):].count(os.sep)
        print("    " * depth + os.path.basename(root) + "/")
        for name in sorted(files):
            print("    " * (depth + 1) + name)


def run_tests(folder):
    """Runs pytest on a project folder, as notebooks/run_tests.py does in the job. Fails the cell if a test fails."""
    for name, module in list(sys.modules.items()):   # forget earlier imports, so file edits are picked up
        if (getattr(module, "__file__", None) or "").startswith(PROJECT):
            del sys.modules[name]
    retcode = pytest.main([os.path.join(PROJECT, folder), "-v", "--tb=short", "-p", "no:cacheprovider"])
    assert retcode == 0, f"pytest returned {retcode}: read the FAILED lines above."


def set_cancelled_statuses(*statuses):
    """Edits the CANCELLED_STATUSES line in src/sales.py, as a developer would by hand."""
    path = os.path.join(PROJECT, "src", "sales.py")
    with open(path) as f:
        text = f.read()
    line = "CANCELLED_STATUSES = " + json.dumps(list(statuses))
    with open(path, "w") as f:
        f.write(re.sub(r"^CANCELLED_STATUSES = .*$", line, text, flags=re.M))
    print("src/sales.py, line 5 →", line)


print(f"Schema:    {CATALOG}.{SCHEMA} (table sales_sample, 5 rows)")
print(f"Project:   {PROJECT}")
print(f"Workspace: {HOST}")

In [0]:
# Project files: the tiny repository that the demos work on. No need to show this cell in class.
FILES = {}

FILES[".gitignore"] = '''
# Bundle deployment state, created by the Databricks CLI
.databricks/
# Python and pytest caches
__pycache__/
.pytest_cache/
# Logs, local secrets, Mac clutter
*.log
.env
.DS_Store
'''

FILES["pytest.ini"] = '''
[pytest]
# Tests import src.sales from the project root
pythonpath = .
'''

FILES["conftest.py"] = '''
"""Shared pytest fixtures. Every test that names `spark` gets the Spark session."""
import pytest
from pyspark.sql import SparkSession


@pytest.fixture(scope="session")
def spark():
    return SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
'''

FILES["src/sales.py"] = '''
"""Business logic of the sales job: plain PySpark, no tables, no widgets, so it is easy to test."""
from pyspark.sql import DataFrame, functions as F

VAT_RATE = 0.255
CANCELLED_STATUSES = ["CANCELLED"]


def remove_cancelled(df: DataFrame) -> DataFrame:
    """Removes cancelled sales. Ignores case and spaces: ' Cancelled ' counts too."""
    return df.where(~F.upper(F.trim("status")).isin(CANCELLED_STATUSES))


def add_gross(df: DataFrame, rate: float = VAT_RATE) -> DataFrame:
    """Adds the column gross: net plus VAT, rounded to cents."""
    return df.withColumn("gross", F.round(F.col("net") * (1 + rate), 2))
'''

FILES["tests/unit/test_sales.py"] = '''
"""Unit tests: one function, invented rows, one assertion. No tables."""
import pytest

from src.sales import add_gross, remove_cancelled


@pytest.fixture
def sales(spark):
    """Three invented sales: we know the right answers, and nothing is sensitive."""
    return spark.createDataFrame(
        [(1, 100.0, "PAID"), (2, 20.0, " cancelled "), (3, 10.0, "Cancelled")],
        "sale_id INT, net DOUBLE, status STRING",
    )


def test_remove_cancelled_ignores_case_and_spaces(sales):
    assert [row.sale_id for row in remove_cancelled(sales).collect()] == [1]


def test_add_gross_adds_vat(sales):
    assert add_gross(sales).where("sale_id = 1").first().gross == 125.5


def test_add_gross_keeps_every_row(sales):
    assert add_gross(sales).count() == 3
'''

FILES["tests/integration/test_sales_table.py"] = '''
"""Integration test: real Spark and a real table in a dedicated test schema. Never production."""
import os

import pytest

from src.sales import add_gross, remove_cancelled


def test_pipeline_on_test_table(spark):
    table = os.environ.get("TEST_TABLE")
    if not table:
        pytest.skip("Set TEST_TABLE to a table in a test schema.")
    result = add_gross(remove_cancelled(spark.read.table(table)))
    assert result.columns == ["sale_id", "net", "status", "gross"]
    assert result.count() == 3   # 5 sample rows, 2 cancelled
'''

FILES["notebooks/run_tests.py"] = '''
# Databricks notebook source
# MAGIC %pip install -q pytest

# COMMAND ----------

# First task of sales_job: the unit tests. A failing test fails this task, so load_sales never runs.
import os
import sys

import pytest

sys.dont_write_bytecode = True
retcode = pytest.main([os.path.abspath("../tests/unit"), "-v", "-p", "no:cacheprovider"])
assert retcode == 0, "Unit tests failed: nothing was loaded."
'''

FILES["notebooks/load_sales.py"] = '''
# Databricks notebook source
# Second task of sales_job: runs only after the unit tests passed.
# catalog and schema arrive as widgets, from base_parameters in resources/sales_job.yml.
import os
import sys

sys.path.append(os.path.abspath(".."))   # the bundle root, so that src/ can be imported
from src.sales import add_gross, remove_cancelled

dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
schema = f"`{dbutils.widgets.get('catalog')}`.`{dbutils.widgets.get('schema')}`"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
sales = spark.createDataFrame(
    [(1, 100.0, "PAID"), (2, 250.0, "paid"), (3, 80.0, " Cancelled ")],
    "sale_id INT, net DOUBLE, status STRING",
)
add_gross(remove_cancelled(sales)).write.mode("overwrite").saveAsTable(f"{schema}.sales_gross")
print(f"Wrote {schema}.sales_gross")
'''

FILES["databricks.yml"] = '''
# Declarative Automation Bundle (formerly Databricks Asset Bundle). The one required file.
bundle:
  name: sales_demo

include:
  - resources/*.yml

variables:
  catalog:
    description: Catalog for the sales table
    default: <CATALOG>
  schema:
    description: Schema for the sales table
    default: m12_demo

targets:
  dev:
    mode: development     # [dev you] prefix, schedules paused
    default: true
    workspace:
      host: <HOST>

  prod:
    mode: production      # exact names, schedules run, stricter checks
    git:
      branch: main        # deploys only from main
    workspace:
      host: <HOST>
      root_path: /Workspace/Users/<USER>/.bundle/${bundle.name}/${bundle.target}
    variables:
      schema: m12_prod
    # run_as:
    #   service_principal_name: <application-id>   # recommended for production
'''

FILES["resources/sales_job.yml"] = '''
resources:
  jobs:
    sales_job:                      # the job key: bundle run and bundle open use it
      name: sales_job
      schedule:                     # the Monday run. Paused in dev mode
        quartz_cron_expression: "0 0 6 ? * MON"
        timezone_id: Europe/Helsinki
      tasks:                        # no cluster defined: runs on serverless
        - task_key: run_unit_tests
          notebook_task:
            notebook_path: ../notebooks/run_tests.py

        - task_key: load_sales
          depends_on:               # runs only when the tests passed
            - task_key: run_unit_tests
          notebook_task:
            notebook_path: ../notebooks/load_sales.py
            base_parameters:        # arrive as widgets in the notebook
              catalog: ${var.catalog}
              schema: ${var.schema}
'''

for path, text in FILES.items():
    text = text.replace("<HOST>", HOST).replace("<USER>", USER).replace("<CATALOG>", CATALOG)
    print(f"{write(path, text):8} {path}")

## Demo 1 · Git folders *(external: GitHub or Azure DevOps)*
**A real Git client in the workspace. One Git folder per developer, pull before you start, keep the repository small.**

Before class: create an **empty repository** in your provider (GitHub: **New repository**, tick **Add a README**).

1. **Credential:** user menu → **Settings** → **Linked accounts** → add your Git credential. It's personal: never a shared token.
2. **Clone:** **Workspace** → your user folder → **Create** → **Git folder** → paste the repository URL → **Create Git folder**.
3. **Move this notebook into the Git folder** (⋮ → **Move**) and run the setup cells again. The project files now sit inside the repository.
4. Run the next cell: the layout that the repository follows.

In [0]:
show_tree()
print()
show(".gitignore")

5. Click the **branch name** next to the notebook title. The Git dialog lists every new file. Message *Add sales demo project* → **Commit & Push**. (Straight to `main` only because it's the first commit.)
6. Click **Pull** in the same dialog. Do it every morning: it brings in what colleagues pushed.

Points to make:
- **One folder per developer.** If two people share a Git folder, one branch switch changes the notebooks for both.
- A pull that changes a notebook clears its outputs. **Commit before you pull.**
- `.gitignore` keeps caches, secrets and `.databricks/` (bundle state) out of Git. It only affects files that Git doesn't track yet.
- Keep the repository small: a branch is capped at 1 GB, and fewer than 20,000 assets is the recommendation. Bigger repositories use sparse checkout.

## Demo 2 · Branches, pull requests, conflicts *(external: your Git provider)*
**Databricks does the Git operations. Your provider does the review.**

### 2a · Branch, change, commit
1. Git dialog → **Create Branch** → `feature/refunds`, based on `main`.
2. Finance wants refunds left out too. Open `sales_demo/src/sales.py` and change line 5 to `CANCELLED_STATUSES = ["CANCELLED", "REFUNDED"]`. Or run the next cell.

In [0]:
set_cancelled_statuses("CANCELLED", "REFUNDED")

3. Git dialog: `src/sales.py` is listed. Click it to **review the diff**: red is removed, green is added. This is where you catch a stray debug print.
4. Message *Leave refunds out of sales* → **Commit & Push**.

### 2b · Pull request: in your provider, never in Databricks
5. In GitHub (**Compare & pull request**) or Azure DevOps (**Create a pull request**): `feature/refunds` → `main`.
6. Review, approve, **Merge**. Delete the branch.
7. Back in Databricks: Git dialog → switch to `main` → **Pull**. Line 5 now includes `"REFUNDED"`.

Steps 1–4 and 7 happen in Databricks, steps 5–6 in the provider. Databricks has no pull request screen.

### 2c · A merge conflict
8. **A colleague** (you, in the provider's web editor, on `main`) changes line 5 of `sales_demo/src/sales.py` to `CANCELLED_STATUSES = ["CANCELLED", "REFUNDED", "VOID"]` and commits.
9. **You**, in Databricks, still on `main` and without pulling: run the next cell. It puts `"REJECTED"` on the same line.

In [0]:
set_cancelled_statuses("CANCELLED", "REFUNDED", "REJECTED")

10. Git dialog → **Pull**. The conflict screen opens on `src/sales.py`:
    ```
    <<<<<<< …     one side
    CANCELLED_STATUSES = ["CANCELLED", "REFUNDED", "REJECTED"]
    =======       separator
    CANCELLED_STATUSES = ["CANCELLED", "REFUNDED", "VOID"]
    >>>>>>> …     the other side
    ```
    The text after `<<<<<<<` and `>>>>>>>` names the two sides.
11. Both changes are right, so **edit by hand**: keep one line, `CANCELLED_STATUSES = ["CANCELLED", "REFUNDED", "VOID", "REJECTED"]`, and delete the marker lines. **Keep all current changes** or **Take all incoming changes** would throw one person's work away.
12. **Mark As Resolved** → **Continue Merge**. If the dialog still lists `src/sales.py`, **Commit & Push**. Made a mess? **Abort** puts everything back.

Fewer conflicts: pull from `main` often, keep branches short-lived, prefer merge over rebase.

## Demo 3 · Testing pyramid *(this notebook only)*
**Many small fast tests, a few slow ones, none against production.**

| Layer | How many | Speed | In this project |
|---|---|---|---|
| Unit | Many | Fast | `tests/unit/test_sales.py`: one function, invented rows |
| Integration | Some | Seconds | `tests/integration/test_sales_table.py`: a real table in the test schema |
| End-to-end | Few | Minutes | The whole job, off-peak. Demo 5 runs it |
| UAT | Varies | People | The business checks the result in staging, against criteria agreed up front |

The logic lives in `src/sales.py`, a plain Python file rather than a notebook, so tests can import it. A **fixture** builds test data for every test that names it: `sales` below, and `spark` in `conftest.py`. Files must be called `test_*.py`, or pytest silently runs nothing.

In [0]:
show("src/sales.py")
show("tests/unit/test_sales.py")

In [0]:
run_tests("tests/unit")

**What a failure looks like.** The next cell adds a test with a wrong expectation (someone still thinks VAT is 24 %), runs the suite and removes the test again. The failing `assert` is what stops a job before anything is loaded.

In [0]:
broken = os.path.join(PROJECT, "tests", "unit", "test_old_vat.py")
with open(broken, "w") as f:
    f.write('''
from src.sales import add_gross

def test_gross_with_old_vat(spark):
    df = spark.createDataFrame([(1, 100.0, "PAID")], "sale_id INT, net DOUBLE, status STRING")
    assert add_gross(df).first().gross == 124.0
''')
try:
    run_tests("tests/unit")
except AssertionError as e:
    print("\n❌ Caught so that Run All continues. Without try/except the cell stops here, and in the job the task fails:", e)
finally:
    os.remove(broken)

**Integration test:** real Spark on the `sales_sample` table in the test schema, never a production table. It checks the schema and the row count: 5 rows in, 2 cancelled, 3 out.

In [0]:
show("tests/integration/test_sales_table.py")
run_tests("tests/integration")

**UAT:** people judge the result in a staging environment. A notebook can check the criterion they agreed before you built anything: *total gross within 1 % of finance's own figure, 490.00 €.*

In [0]:
import importlib
import src.sales as sales_code
importlib.reload(sales_code)   # picks up edits to src/sales.py

FINANCE_TOTAL = 490.00         # from finance's own spreadsheet
result = sales_code.add_gross(sales_code.remove_cancelled(spark.read.table(TEST_TABLE)))
ours = result.agg(F.sum("gross")).first()[0]
difference = abs(ours - FINANCE_TOTAL) / FINANCE_TOTAL

display(result)
print(f"Pipeline {ours:.2f} € vs finance {FINANCE_TOTAL:.2f} €: {difference:.2%} apart → "
      + ("✅ accepted" if difference <= 0.01 else "❌ rejected"))

## Demo 4 · Bundle configuration *(this notebook only)*
**One YAML project describes the job once. Variables change values, targets decide where and how it lands.**

Declarative Automation Bundles were called Databricks Asset Bundles. Same `databricks bundle` command, same YAML.

- `databricks.yml` is the one required file, and `bundle: name:` the only required key. `include:` merges the files in `resources/`.
- `${var.schema}` is filled in at **deploy** time, not run time. Strongest first: `--var="schema=x"` → environment variable `BUNDLE_VAR_schema` → the target's `variables:` → `default:`.
- **dev** (`mode: development`): names get the prefix `[dev <you>]`, schedules are paused, concurrent runs are allowed. Five developers, five jobs, no collisions.
- **prod** (`mode: production`): exact names, schedules run, `root_path` is required so only one copy exists, `git: branch: main` refuses other branches. A service principal in `run_as` is recommended.

In [0]:
show("databricks.yml")

In [0]:
show("resources/sales_job.yml")

- `sales_job` (the key) is the handle for `bundle run` and `bundle open`. `name:` is what the Jobs UI shows.
- No cluster anywhere: the tasks run on serverless.
- `depends_on`: `load_sales` only runs when `run_unit_tests` passed. That's the "test before promote" rule as configuration.
- `base_parameters` arrive in `notebooks/load_sales.py` as widgets. Paths are relative to this YAML file.
- A `spark_python_task` with `python_file:` runs a plain `.py` file instead of a notebook.

## Demo 5 · Deploy with the Databricks CLI *(external: your laptop)*
**Validate, plan, deploy, verify. Read the plan before you deploy.**

Get the project onto your laptop: `git clone <your repository URL>`, or **Code** → **Download ZIP** in GitHub. Then, in PowerShell with a large font:

| Step | Command | What to point at |
|---|---|---|
| Version | `databricks -v` | Any 1.x is current |
| Sign in | `databricks auth login --host <workspace URL from setup>` | OAuth in the browser, no personal access token |
| Go to the bundle | `cd <repository>\sales_demo` | The folder with `databricks.yml` |
| Validate | `databricks bundle validate` | Bundle name, target `dev`, host, `Validation OK!` |
| Plan | `databricks bundle plan` | `create jobs.sales_job`. An unexpected *delete* means stop |
| Deploy | `databricks bundle deploy -t dev` | The only command that changes the workspace |
| Verify | `databricks bundle summary` | Links to the deployed job |
| Run (end-to-end) | `databricks bundle run sales_job` | Tests first, then the load. A few minutes |
| Open | `databricks bundle open sales_job` | `[dev you] sales_job`, schedule **paused** |

When it goes wrong:
- **Authentication error:** `databricks auth login` again.
- **Lock held:** `--force-lock`, only when nobody else is deploying.
- **Jobs running:** production pipelines add `--fail-on-active-runs`. CI/CD adds `--auto-approve`.
- **Unknown field warning:** update the CLI, then validate again.
- **No CLI on this laptop:** open `databricks.yml` in the Git folder and deploy from the **Deployments** panel in the workspace.

After the deploy, run the next cell.

In [0]:
# What development mode created
jobs = [j for j in w.jobs.list() if j.settings and j.settings.name.endswith("sales_job")]
for job in jobs:
    schedule = job.settings.schedule
    status = schedule.pause_status.value if schedule and schedule.pause_status else "no schedule"
    print(f"{job.settings.name}  |  schedule: {status}  |  job id: {job.job_id}")
if not jobs:
    print("No sales_job yet: run `databricks bundle deploy -t dev` on your laptop.")

# After `databricks bundle run sales_job`: the table that the load task wrote
try:
    display(spark.table(f"`{CATALOG}`.`{SCHEMA}`.sales_gross"))
except Exception:
    print("No sales_gross table yet: run `databricks bundle run sales_job`.")

## Clean up (after the course)
- Laptop: `databricks bundle destroy` removes the dev job and its deployed files.
- Provider: delete merged branches. Workspace: delete the Git folder if you don't need it any more.
- Then uncomment and run the next cell.

In [0]:
# spark.sql(f"DROP SCHEMA IF EXISTS `{CATALOG}`.`{SCHEMA}` CASCADE")